# The matched fractional-step run

Marches the projection solver over the **same span, from the same field, on the
same grid** as the least-squares DNS, so that the only difference between the two
records is the discretisation.

| | least squares | this run |
|---|---|---|
| starting field | run01 checkpoint at $t=4.96$ | **the same field**, converted |
| span | $t = 4.96 \to 30$ | $t = 4.96 \to 30$ |
| grid | 6×18 elements, $N=8$, 32 modes | identical |
| $\Delta t$ | 8e−4 | 8e−4 |
| averaging window | $t = 5.2\to30$ | $t = 5.2\to30$ |
| statistics collector | `minchan_stats.py` | its direct ancestor, same schema |

**Why the same field matters.** Starting each code from its own archived state is
already fair statistically, but the two would then sample different realisations
of the same flow, and a 1–3 % difference could be the realisation rather than the
scheme. Marching the identical field removes that objection.

**Cost:** 31,299 steps at ~1.92 s — about **16.7 hours**, so two sessions.
The least-squares run took 14.1 h for the same span.


In [ ]:
#@title 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
#@title 2. Code
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin main && git reset -q --hard origin/main
else:
    !git clone -q --branch main https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q numba scipy matplotlib


In [ ]:
#@title 3. Drive, and the seed
#@markdown If the mount fails, this retries with force_remount, which clears a
#@markdown stale mount from an earlier session -- the usual cause after a runtime
#@markdown restart.  If it still fails, see the notes under this cell.
from google.colab import drive
import os, numpy as np
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount failed ({e}); retrying with force_remount', flush=True)
    drive.mount('/content/drive', force_remount=True)

DRIVE = '/content/drive/MyDrive/lssem_fs'    #@param {type:"string"}
LS_CKPT = '/content/drive/MyDrive/lssem_data/checkpoint_0006200.npz'  #@param {type:"string"}
OUT = '/content/fs_run'
assert os.path.isdir('/content/drive/MyDrive'), 'Drive did not mount'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(LS_CKPT):
    raise SystemExit(f'no least-squares checkpoint at {LS_CKPT} -- '
                     f'check the path, or upload it to MyDrive/lssem_data/')

SEED = f'{DRIVE}/fs_seed_from_fosls.npz'
if not os.path.exists(SEED):
    print('converting the least-squares checkpoint into a projection restart\n')
    !python colab/fs_seed_from_fosls.py --ckpt {LS_CKPT} --out {SEED}
else:
    with np.load(SEED) as z: print(f'seed already on Drive at t = {float(z["t"]):.4f}')
ck = f'{DRIVE}/chk_latest.npz'
if os.path.exists(ck):
    with np.load(ck) as z: print(f'\nrun in progress: t = {float(z["t"]):.4f} of 30')


**If the mount still fails.** It is a Colab/Drive problem, not this code:

1. **Runtime → Restart session**, then run cells 1–3 again. This clears a stale
   mount, which is the most common cause after an earlier session ended badly.
2. Watch for the **authorization popup** — if your browser blocks pop-ups, or the
   window is dismissed before you pick an account, the mount reports exactly this
   `ValueError: mount failed`.
3. Try mounting in a cell on its own: `from google.colab import drive;
   drive.mount('/content/drive', force_remount=True)`. Errors are clearer without
   anything else in the cell.
4. Check your Drive is not full — the run writes ~10 MB per checkpoint plus a few
   kB per statistics snapshot.

**You can start the run without Drive** by setting `DRIVE = '/content/fs_run'` and
`SEED` to a local path, but then nothing survives the VM, and this campaign needs
two sessions — so it is worth fixing the mount first.


In [ ]:
#@title 4. THE RUN
HOURS    = 10.0   #@param {type:"number"}
TARGET_T = 30.0   #@param {type:"number"}
#@markdown Two sessions at ~1.92 s/step.  It stops itself at the budget with a
#@markdown checkpoint on Drive, and the accumulators carry across restarts, so a
#@markdown campaign split over nights averages exactly as one run would.
cmd = (f'python -u colab/run_fs_dns.py --hours {HOURS} --target-t {TARGET_T} '
       f'--dt 8e-4 --chkmin 5 --out {OUT} --drive {DRIVE} --seed {SEED} '
       f'--backend torch')
print(cmd, flush=True)
!{cmd}


In [ ]:
#@title 5. Statistics over the matched window, against the same databases
#@markdown `section10.py` unchanged: the projection collector writes the same
#@markdown schema.  The window starts at t = 5.2, discarding the same quarter
#@markdown turnover the least-squares run discarded.
import glob, numpy as np, os
snaps = sorted(glob.glob(f'{DRIVE}/stats_t*.npz'))
print(f'{len(snaps)} snapshots on Drive')
if snaps:
    def t_of(f):
        with np.load(f, allow_pickle=True) as z: return float(z['t'])
    A = min(snaps, key=lambda f: abs(t_of(f) - 5.2))
    B = snaps[-1]
    print(f'window {t_of(A):.2f} -> {t_of(B):.2f}')
    !python colab/section10.py {A} {B} --out {DRIVE} --label "fractional step (E)"
    from IPython.display import Image, display
    display(Image(f'{DRIVE}/section10_profiles.png'))


## Reading the comparison

Both runs are now scored by the same script against the same five databases, over
the same window, from the same starting field. The differences that remain are
the discretisation and nothing else.

**What to expect from the earlier evidence.** The low-order statistics should
agree closely — the operator choice did not move them in either previous A/B, and
the weak divergence is what the momentum equation sees. The interesting columns
are the ones that differentiate the velocity field, and the pointwise divergence,
where the two methods differ by three orders of magnitude and place their
residuals in different places (`DIVERGENCE_CONSEQUENCES.md`).

**If the statistics agree**, that is the result: at matched cost and matched
conditions, the two formulations produce the same turbulence, and the case for
either rests on what it gives you besides the mean profile.
